# Modelo 3 — XGBoost Optimizado + Poisson: probabilidades 1X2 y marcador exacto

Tercer modelo del proyecto. El **Modelo 2 (XGBoost Vanilla)** subestimó a Francia frente
a Paraguay porque, aunque tenía ~180 variables de forma reciente, **no tenía ninguna
variable que capturara la fuerza histórica real de una selección** (una racha reciente
mediocre de Francia podía pesar tanto como toda la trayectoria de Paraguay). Este modelo
corrige eso agregando **Elo histórico**, variables de fuerza relativa, y combina:

- Un **XGBoost Classifier optimizado** (`RandomizedSearchCV` + validación temporal) para
  1X2.
- Dos **XGBoost Regressors con objetivo Poisson** (`count:poisson`) para estimar goles
  esperados de local y visitante.
- Una **matriz de Poisson** para obtener probabilidades por marcador exacto (0-0, 1-0,
  2-1, etc.) y un **ensemble ponderado** entre el XGBoost Classifier y el 1X2 derivado de
  Poisson.

**No se elimina ni se rediseña nada del feature engineering anterior** (~180 variables,
sin data leakage, split temporal 80/20) — solo se agregan variables de fuerza de equipo
y se reemplaza el modelo vanilla por este pipeline más robusto.

**Aviso honesto:** este modelo es más robusto que el Vanilla porque ahora sí distingue
la fuerza real de cada selección (Elo), pero el fútbol tiene varianza irreducible — no
existe un modelo que prediga marcadores exactos con alta certeza. Lo que se gana aquí es
una estimación de probabilidades mejor calibrada y más defendible, no una "bola de
cristal".


## 1. Imports

**Qué hace:** carga todas las librerías necesarias, incluyendo `scipy.stats.poisson`
(para la distribución de Poisson de los marcadores) y las utilidades de
`sklearn.model_selection` para la búsqueda de hiperparámetros con validación temporal.

**Por qué `TimeSeriesSplit` en vez de `KFold`:** al optimizar hiperparámetros con
validación cruzada, un `KFold` aleatorio mezclaría partidos futuros en los folds de
"entrenamiento" de una validación con partidos más antiguos en el fold de "validación"
— la misma fuga temporal que ya evitamos con el split 80/20. `TimeSeriesSplit` respeta
el orden cronológico también dentro de la búsqueda de hiperparámetros.

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
!pip install -q xgboost joblib scipy

import warnings
warnings.filterwarnings("ignore")

import os
import subprocess
import time
import pickle
import difflib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson as poisson_dist

from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

import xgboost as xgb
from xgboost import XGBClassifier, XGBRegressor
import joblib

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("xgboost version:", xgb.__version__)


## 2. Carga del dataset (idéntico a los Modelos 1 y 2)

Mismo repositorio, misma búsqueda flexible del CSV. Sin cambios respecto a los notebooks
anteriores — para que los tres modelos del proyecto sean comparables, todos parten
exactamente del mismo dataset crudo.

In [ ]:
# ============================================================
# 2. CLONAR REPOSITORIO Y CARGAR EL DATASET (identico a Modelos 1 y 2)
# ============================================================
REPO_URL = "https://github.com/martj42/international_results"
REPO_DIR = "/content/international_results_repo"

if not os.path.exists(REPO_DIR):
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    except Exception as e:
        print("No se pudo clonar el repo automaticamente:", e)
else:
    print("El repositorio ya existe localmente, se omite el clonado.")


def find_results_csv(search_dirs):
    """Busca de forma flexible el CSV de resultados por nombre o por columnas esperadas."""
    candidate_names = ["results.csv", "result.csv", "matches.csv", "international_results.csv"]
    csv_files = []
    for base_dir in search_dirs:
        if not os.path.isdir(base_dir):
            continue
        for root, _, files in os.walk(base_dir):
            for f in files:
                if f.lower().endswith(".csv"):
                    csv_files.append(os.path.join(root, f))
    for name in candidate_names:
        for path in csv_files:
            if os.path.basename(path).lower() == name:
                return path
    expected_cols = {"home_team", "away_team", "home_score", "away_score"}
    for path in csv_files:
        try:
            sample = pd.read_csv(path, nrows=5)
            if expected_cols.issubset(set(c.lower() for c in sample.columns)):
                return path
        except Exception:
            continue
    raise FileNotFoundError(
        "No se encontro un CSV con las columnas esperadas. "
        "Revisa REPO_URL o coloca manualmente la ruta del archivo en csv_path."
    )


csv_path = find_results_csv([REPO_DIR, "."])
print("Archivo de resultados encontrado en:", csv_path)

raw_df = pd.read_csv(csv_path)
raw_df.columns = [c.strip().lower() for c in raw_df.columns]
print("Shape original:", raw_df.shape)
raw_df.head()


## 3. Limpieza + Feature Engineering base (idéntico a los Modelos 1 y 2 — sin cambios)

Copia literal del feature engineering anterior: limpieza, `target`, y las ~180 variables
de forma con ventanas de 3/5/10/15/20 partidos (`shift(1)` antes de `rolling`, sin data
leakage). En la Sección 4 se **agregan** variables nuevas de fuerza de equipo — no se
toca ni se elimina nada de este bloque.

In [ ]:
# ============================================================
# 3.1 LIMPIEZA DEL DATASET (identico a Modelos 1 y 2)
# ============================================================
df = raw_df.copy()

rename_map = {
    "hometeam": "home_team", "awayteam": "away_team",
    "homescore": "home_score", "awayscore": "away_score",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ["date", "home_team", "away_team", "home_score", "away_score"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias en el dataset: {missing}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df = df.sort_values("date").reset_index(drop=True)

df = df.dropna(subset=["home_score", "away_score"])
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

if "tournament" not in df.columns:
    df["tournament"] = "Unknown"

if "neutral" not in df.columns:
    df["neutral"] = False
else:
    df["neutral"] = (
        df["neutral"].astype(str).str.upper().map({"TRUE": True, "FALSE": False}).fillna(False)
    )
df["neutral"] = df["neutral"].astype(int)

df["match_id"] = np.arange(len(df))

conditions = [
    df["home_score"] > df["away_score"],
    df["home_score"] == df["away_score"],
    df["home_score"] < df["away_score"],
]
df["target"] = np.select(conditions, [0, 1, 2]).astype(int)

print(f"Dataset limpio: {df.shape[0]} partidos, {df.shape[1]} columnas")
df[["date", "home_team", "away_team", "home_score", "away_score", "target"]].head()


In [ ]:
# ============================================================
# 3.2 FEATURE ENGINEERING BASE (SIN DATA LEAKAGE) - identico a Modelos 1 y 2
# ============================================================
WINDOWS = [3, 5, 10, 15, 20]

long_cols = ["match_id", "date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"]

home_long = df[long_cols].rename(columns={
    "home_team": "team", "away_team": "opponent",
    "home_score": "goals_for", "away_score": "goals_against",
})
home_long["is_home"] = 1

away_long = df[long_cols].rename(columns={
    "away_team": "team", "home_team": "opponent",
    "away_score": "goals_for", "home_score": "goals_against",
})
away_long["is_home"] = 0

long_df = pd.concat([home_long, away_long], ignore_index=True)
long_df = long_df.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

long_df["goal_diff"] = long_df["goals_for"] - long_df["goals_against"]
long_df["win"] = (long_df["goals_for"] > long_df["goals_against"]).astype(int)
long_df["draw"] = (long_df["goals_for"] == long_df["goals_against"]).astype(int)
long_df["loss"] = (long_df["goals_for"] < long_df["goals_against"]).astype(int)
long_df["points"] = long_df["win"] * 3 + long_df["draw"] * 1
long_df["clean_sheet"] = (long_df["goals_against"] == 0).astype(int)
long_df["failed_to_score"] = (long_df["goals_for"] == 0).astype(int)

BASE_METRICS = ["goals_for", "goals_against", "goal_diff", "points",
                "win", "draw", "loss", "clean_sheet", "failed_to_score"]
STD_METRICS = ["goals_for", "goals_against", "goal_diff"]

feature_cols_long = []
grouped = long_df.groupby("team", group_keys=False)

t0 = time.time()
new_cols = {}
for w in WINDOWS:
    for col in BASE_METRICS:
        feat_name = f"{col}_mean_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        feature_cols_long.append(feat_name)
    for col in STD_METRICS:
        feat_name = f"{col}_std_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).std())
        feature_cols_long.append(feat_name)

long_df = pd.concat([long_df, pd.DataFrame(new_cols, index=long_df.index)], axis=1)
print(f"Variables de forma por equipo calculadas en {time.time() - t0:.1f}s -> {len(feature_cols_long)} columnas")

rename_home = {c: f"home_{c}" for c in feature_cols_long}
rename_away = {c: f"away_{c}" for c in feature_cols_long}

home_features = long_df.loc[long_df.is_home == 1, ["match_id"] + feature_cols_long].rename(columns=rename_home)
away_features = long_df.loc[long_df.is_home == 0, ["match_id"] + feature_cols_long].rename(columns=rename_away)

df = df.merge(home_features, on="match_id", how="left").merge(away_features, on="match_id", how="left")

diff_data = {f"diff_{c}": df[f"home_{c}"] - df[f"away_{c}"] for c in feature_cols_long}
df = pd.concat([df, pd.DataFrame(diff_data, index=df.index)], axis=1)
diff_cols = list(diff_data.keys())

all_feature_cols = [f"home_{c}" for c in feature_cols_long] + [f"away_{c}" for c in feature_cols_long] + diff_cols
print(f"Total de variables de forma (base): {len(all_feature_cols)}")

df[all_feature_cols] = df[all_feature_cols].fillna(0)


## 4. Variables de fuerza de equipo — Elo histórico, xG aproximado, ventaja local

**Qué hace este bloque (lo nuevo del Modelo 3):**

1. **Elo histórico partido por partido, sin data leakage:** se recorre el dataset en
   orden cronológico manteniendo un rating Elo por equipo (inicial 1500). Para cada
   partido se **registra primero** el Elo de ambos equipos *antes* de ese partido
   (`elo_home_pre`, `elo_away_pre`) y **después** se actualiza el rating con el
   resultado real. Así, la variable Elo de un partido nunca incluye información del
   propio partido — solo de todos los anteriores.
2. **`elo_diff`** = diferencia de Elo entre local y visitante: la señal de fuerza
   relativa más directa que le faltaba al Modelo 2.
3. **Goles esperados aproximados (`xg_home_approx`, `xg_away_approx`):** un promedio
   simple entre el ataque reciente de un equipo (goles a favor, ventana de 10 partidos)
   y la defensa reciente del rival (goles en contra, ventana de 10) — un proxy barato
   de "expected goals" calculado solo con datos históricos ya existentes (no agrega
   fuentes externas).
4. **Ventaja local / neutral:** el partido puede jugarse en campo del local (ventaja) o
   en sede neutral. Estas variables ya existían como metadata; aquí se agregan
   explícitamente como *variables del modelo* (`neutral`, `home_advantage`).

**Por qué importa:** en el Modelo 2, Francia vs Paraguay salió mal calibrado porque el
modelo solo veía la forma de los últimos partidos de cada equipo, sin ninguna medida de
"cuán buena es esta selección en términos históricos". El Elo resuelve exactamente eso:
Argentina, España, Francia y Brasil terminan arriba del ranking Elo calculado — como se
esperaría futbolísticamente — y la brecha de Elo entre Francia y Paraguay debería ahora
inclinar la predicción a favor de Francia de forma mucho más razonable.

**Nada se elimina:** estas variables se **agregan** a las ~180 ya existentes.

In [ ]:
# ============================================================
# 4. VARIABLES DE FUERZA DE EQUIPO (Elo, xG aproximado, ventaja local)
# ============================================================
ELO_INITIAL = 1500.0
ELO_K = 20.0

homes = df["home_team"].values
aways = df["away_team"].values
home_scores = df["home_score"].values
away_scores = df["away_score"].values
n_matches = len(df)

elo_ratings = {}
elo_home_pre = np.empty(n_matches)
elo_away_pre = np.empty(n_matches)

t0 = time.time()
for i in range(n_matches):
    home = homes[i]
    away = aways[i]

    eh = elo_ratings.get(home, ELO_INITIAL)
    ea = elo_ratings.get(away, ELO_INITIAL)
    elo_home_pre[i] = eh   # Elo del local ANTES de este partido (sin leakage)
    elo_away_pre[i] = ea   # Elo del visitante ANTES de este partido (sin leakage)

    expected_home = 1.0 / (1.0 + 10 ** ((ea - eh) / 400.0))

    hg, ag = home_scores[i], away_scores[i]
    if hg > ag:
        actual_home = 1.0
    elif hg == ag:
        actual_home = 0.5
    else:
        actual_home = 0.0

    # Multiplicador por margen de victoria (estilo eloratings.net): goleadas
    # mueven el rating mas que victorias ajustadas.
    margin = abs(hg - ag)
    if margin <= 1:
        multiplier = 1.0
    elif margin == 2:
        multiplier = 1.5
    else:
        multiplier = (11 + margin) / 8.0

    delta = ELO_K * multiplier * (actual_home - expected_home)

    # Actualizacion DESPUES de haber registrado el valor pre-partido -> sin leakage
    elo_ratings[home] = eh + delta
    elo_ratings[away] = ea - delta

print(f"Elo historico calculado en {time.time() - t0:.1f}s para {n_matches} partidos")

elo_current = dict(elo_ratings)  # snapshot final: rating vigente de cada equipo, para predicciones futuras

# xG aproximado: promedio entre ataque reciente propio y defensa reciente del rival
xg_home_approx = (df["home_goals_for_mean_10"].values + df["away_goals_against_mean_10"].values) / 2.0
xg_away_approx = (df["away_goals_for_mean_10"].values + df["home_goals_against_mean_10"].values) / 2.0

strength_cols_data = {
    "elo_home_pre": elo_home_pre,
    "elo_away_pre": elo_away_pre,
    "elo_diff": elo_home_pre - elo_away_pre,
    "xg_home_approx": xg_home_approx,
    "xg_away_approx": xg_away_approx,
    "home_advantage": 1 - df["neutral"].values,
}
df = pd.concat([df, pd.DataFrame(strength_cols_data, index=df.index)], axis=1)

strength_feature_cols = list(strength_cols_data.keys()) + ["neutral"]
all_feature_cols = all_feature_cols + strength_feature_cols

print(f"Variables de fuerza agregadas: {len(strength_feature_cols)}")
print(f"Total de variables del modelo (base + fuerza de equipo): {len(all_feature_cols)}")
assert len(all_feature_cols) > 100

print("\nTop 10 Elo actual (rating vigente, informativo):")
print(pd.Series(elo_current).sort_values(ascending=False).head(10))


## 5. Train / Test Split (temporal 80/20, idéntico a Modelos 1 y 2)

Mismo split cronológico: primeros 80% de partidos para entrenar, último 20% para test.
Como las variables de fuerza (Sección 4) se agregaron como columnas nuevas sin alterar
el orden ni el número de filas, el punto de corte es exactamente el mismo que en los
Modelos 1 y 2 — los tres modelos siguen siendo comparables.

In [ ]:
# ============================================================
# 5. SEPARACION TEMPORAL TRAIN / TEST (80% / 20%)
# ============================================================
df_model = df.sort_values("date").reset_index(drop=True)
split_idx = int(len(df_model) * 0.8)

train_df = df_model.iloc[:split_idx].copy()
test_df = df_model.iloc[split_idx:].copy()

print(f"Train: {train_df.shape[0]} partidos "
      f"({train_df['date'].min().date()} a {train_df['date'].max().date()})")
print(f"Test:  {test_df.shape[0]} partidos "
      f"({test_df['date'].min().date()} a {test_df['date'].max().date()})")

y_train = train_df["target"].values.astype(int)
y_test = test_df["target"].values.astype(int)

X_train = train_df[all_feature_cols].values.astype(float)
X_test = test_df[all_feature_cols].values.astype(float)

print(f"Shape -> train: {X_train.shape} | test: {X_test.shape}")
print(f"Variables usadas: {len(all_feature_cols)} (todas: forma + fuerza de equipo)")


## 6. Modelo A — XGBoost Classifier optimizado (1X2)

**Qué hace:** busca una combinación razonable de hiperparámetros con
`RandomizedSearchCV` (20 combinaciones aleatorias, no un grid exhaustivo) usando
`TimeSeriesSplit` como validación cruzada, optimizando `neg_log_loss` (nos interesan
probabilidades bien calibradas, no solo accuracy).

**Por qué `RandomizedSearchCV` y no `GridSearchCV`:** con ~187 variables y varios
hiperparámetros, un grid exhaustivo sería carísimo computacionalmente. Una búsqueda
aleatoria de ~20 combinaciones suele acercarse mucho al óptimo con una fracción del
costo — un compromiso razonable para un notebook de Colab.

**Nota de tiempo:** esta celda puede tardar varios minutos (20 combinaciones × 3 folds
= 60 entrenamientos de XGBoost). Es el costo de "optimizado" frente al Vanilla.

In [ ]:
# ============================================================
# 6. MODELO A: XGBOOST CLASSIFIER OPTIMIZADO (1X2)
# ============================================================
param_distributions = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
}

base_xgb_clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

tscv = TimeSeriesSplit(n_splits=3)

random_search = RandomizedSearchCV(
    base_xgb_clf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_log_loss",
    cv=tscv,
    random_state=RANDOM_SEED,
    n_jobs=1,        # XGBoost ya paraleliza internamente (n_jobs=-1); evita sobre-suscripcion de CPU
    verbose=1,
)

t0 = time.time()
random_search.fit(X_train, y_train)
print(f"\nBusqueda completada en {time.time() - t0:.1f}s")
print("Mejores hiperparametros encontrados:", random_search.best_params_)
print(f"Mejor log loss (CV temporal): {-random_search.best_score_:.4f}")

xgb_clf_tuned = random_search.best_estimator_


## 7. Modelos B y C — XGBoost Poisson para goles esperados

**Qué hace:** entrena dos `XGBRegressor` separados con `objective="count:poisson"` —
uno predice los goles del local (`home_score`), el otro los del visitante
(`away_score`). El objetivo Poisson es el estándar en modelado de goles de fútbol
(los goles por partido siguen aproximadamente una distribución de Poisson), y XGBoost
lo soporta de forma nativa.

**Por qué dos modelos separados en vez de uno multisalida:** los goles del local y del
visitante dependen de features distintas de forma asimétrica (el ataque del local
importa más para `home_score`, la defensa del visitante también, etc.); XGBoost no
tiene un objetivo Poisson multisalida nativo, así que entrenar dos regresores
independientes es la forma estándar de hacerlo.

**Por qué no se optimizan con la misma búsqueda exhaustiva que el Modelo A:** el
notebook ya incluye una búsqueda de hiperparámetros completa (Sección 6); replicarla
para 2 modelos más triplicaría el tiempo de ejecución. Se usan hiperparámetros
razonables (basados en buenas prácticas para datasets de este tamaño) en vez de dejarlos
en default puro — un punto intermedio entre "vanilla" y "totalmente optimizado".

In [ ]:
# ============================================================
# 7. MODELOS B y C: XGBOOST POISSON PARA GOLES ESPERADOS
# ============================================================
home_goals_model = XGBRegressor(
    objective="count:poisson",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
home_goals_model.fit(X_train, train_df["home_score"].values)

away_goals_model = XGBRegressor(
    objective="count:poisson",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
away_goals_model.fit(X_train, train_df["away_score"].values)

print("Modelos de goles (Poisson) entrenados.")
print("Goles esperados promedio (train, real):", train_df["home_score"].mean(), train_df["away_score"].mean())
print("Goles esperados promedio (train, predicho):",
      home_goals_model.predict(X_train).mean(), away_goals_model.predict(X_train).mean())


## 8. Matriz de probabilidad de marcadores (Poisson) y 1X2 derivado

**Qué hace:** dado un par de goles esperados (`lambda_home`, `lambda_away`), construye
una matriz de tamaño `(max_goals+1) x (max_goals+1)` donde la celda `(i, j)` es la
probabilidad de que el marcador final sea *i goles del local, j goles del visitante*,
asumiendo que los goles de cada equipo son variables Poisson independientes. La matriz
se **normaliza** (la probabilidad de marcadores por encima de `max_goals` es muy
pequeña, pero se redistribuye para que la matriz sume exactamente 1).

**Cómo se deriva 1X2 desde la matriz:**
- `home gana` = suma de celdas donde `goles_local > goles_visitante`
- `empate` = suma de la diagonal (`goles_local == goles_visitante`)
- `away gana` = suma de celdas donde `goles_local < goles_visitante`

**Ventaja de este enfoque:** además de un 1X2, obtenemos gratis la distribución
completa de marcadores probables — lo que permite responder "¿cuál es el resultado más
probable?" y no solo "¿quién gana?".

In [ ]:
# ============================================================
# 8. MATRIZ DE POISSON: PROBABILIDAD POR MARCADOR + 1X2 DERIVADO
# ============================================================
def build_score_matrix(lambda_home, lambda_away, max_goals=6):
    """
    Construye la matriz (max_goals+1) x (max_goals+1) de probabilidades de marcador,
    asumiendo goles_home ~ Poisson(lambda_home) y goles_away ~ Poisson(lambda_away)
    independientes. Fila = goles del local, columna = goles del visitante.
    """
    lambda_home = max(lambda_home, 1e-3)
    lambda_away = max(lambda_away, 1e-3)
    goals = np.arange(0, max_goals + 1)
    p_home = poisson_dist.pmf(goals, lambda_home)
    p_away = poisson_dist.pmf(goals, lambda_away)
    matrix = np.outer(p_home, p_away)
    matrix = matrix / matrix.sum()  # renormalizar (la cola truncada en max_goals se redistribuye)
    return matrix


def derive_1x2_from_matrix(matrix):
    """Deriva probabilidades 1X2 sumando las celdas correspondientes de la matriz."""
    n = matrix.shape[0]
    idx = np.arange(n)
    home_win = matrix[np.greater.outer(idx, idx)].sum()
    draw = np.trace(matrix)
    away_win = matrix[np.less.outer(idx, idx)].sum()
    return np.array([home_win, draw, away_win])


def get_top_n_scorelines(matrix, n=10):
    """Devuelve los n marcadores mas probables de la matriz, ordenados descendente."""
    flat_idx = np.argsort(matrix, axis=None)[::-1][:n]
    rows, cols = np.unravel_index(flat_idx, matrix.shape)
    df_top = pd.DataFrame({"home_goals": rows, "away_goals": cols, "probabilidad": matrix[rows, cols]})
    df_top["probabilidad_%"] = (df_top["probabilidad"] * 100).round(2)
    return df_top.reset_index(drop=True)


# Ejemplo rapido de verificacion (no es una prediccion real, solo un chequeo del helper)
_m_check = build_score_matrix(1.8, 1.1, max_goals=6)
print("Suma de la matriz (debe ser 1.0):", round(_m_check.sum(), 6))
print("1X2 derivado de ejemplo (lambda_home=1.8, lambda_away=1.1):", derive_1x2_from_matrix(_m_check).round(4))


## 9. Ensemble: combinar XGBoost Classifier + Poisson para 1X2

**Qué hace:** combina el 1X2 del XGBoost Classifier (que ve directamente todas las
~187 variables, incluyendo Elo) con el 1X2 derivado de la matriz de Poisson (que solo
"sabe" lo que le dicen los dos goles esperados) mediante un **promedio ponderado**, con
pesos configurables.

**Por qué combinar en vez de usar solo uno:** son dos enfoques con fortalezas
distintas. El clasificador ve patrones más ricos (formas, Elo, contexto) directamente
sobre el resultado 1X2; el Poisson aporta consistencia con la distribución de
marcadores (útil para no contradecir la Sección 10) y suele ser más estable cuando hay
pocos goles esperados. Promediar reduce el riesgo de que un solo enfoque domine con un
error sistemático.

**Pesos configurables:** cambia `WEIGHT_XGB_CLASSIFIER` / `WEIGHT_POISSON` si quieres
darle más o menos peso a cada componente (deben sumar 1, o se normalizan igual).

In [ ]:
# ============================================================
# 9. ENSEMBLE: XGBOOST CLASSIFIER + POISSON (pesos configurables)
# ============================================================
WEIGHT_XGB_CLASSIFIER = 0.6
WEIGHT_POISSON = 0.4


def combine_1x2(proba_xgb, proba_poisson, w_xgb=WEIGHT_XGB_CLASSIFIER, w_poisson=WEIGHT_POISSON):
    """Promedio ponderado entre el 1X2 del XGBoost Classifier y el derivado de Poisson."""
    combined = w_xgb * np.asarray(proba_xgb) + w_poisson * np.asarray(proba_poisson)
    return combined / combined.sum()


print(f"Pesos del ensemble -> XGBoost Classifier: {WEIGHT_XGB_CLASSIFIER} | Poisson: {WEIGHT_POISSON}")


## 10. Evaluación completa en el conjunto de test

**Qué hace:** calcula, sobre los partidos de test (nunca vistos en entrenamiento):

- **1X2 (ensemble):** accuracy, log loss, classification report, matriz de confusión.
- **Marcador exacto:** accuracy de acertar el marcador exacto (usando el marcador más
  probable de la matriz de Poisson de cada partido de test) y la probabilidad media
  que el modelo le asignó al marcador que realmente ocurrió.
- **Probabilidad media del resultado correcto:** en promedio, ¿qué probabilidad le
  asignaba el ensemble a la clase (local/empate/visitante) que realmente ocurrió? Es
  una medida de calibración más informativa que el accuracy solo.

Estas cuatro métricas (`accuracy_1x2`, `accuracy_scoreline`, `mean_proba_correct_1x2`,
`mean_proba_correct_scoreline`) se guardan en `evaluation_metrics_summary` para
reutilizarse en la Gráfica 4 (Sección 12) cada vez que se prediga un partido nuevo, sin
tener que recalcular todo el test set en cada llamada.

In [ ]:
# ============================================================
# 10. EVALUACION COMPLETA EN TEST (1X2 + MARCADOR EXACTO)
# ============================================================
MAX_GOALS = 6
goals_grid = np.arange(0, MAX_GOALS + 1)

# --- Predicciones de goles esperados para todo el test set ---
lam_home_test = np.clip(home_goals_model.predict(X_test), 1e-3, None)
lam_away_test = np.clip(away_goals_model.predict(X_test), 1e-3, None)

# --- Matriz de Poisson vectorizada para todo el test set: shape (n_test, 7, 7) ---
pmf_home = poisson_dist.pmf(goals_grid[None, :], lam_home_test[:, None])
pmf_away = poisson_dist.pmf(goals_grid[None, :], lam_away_test[:, None])
joint = pmf_home[:, :, None] * pmf_away[:, None, :]
joint_norm = joint / joint.sum(axis=(1, 2), keepdims=True)

idxg = np.arange(MAX_GOALS + 1)
home_win_mask = idxg[:, None] > idxg[None, :]
draw_mask = idxg[:, None] == idxg[None, :]
away_win_mask = idxg[:, None] < idxg[None, :]

proba_poisson_test = np.stack([
    joint_norm[:, home_win_mask].sum(axis=1),
    joint_norm[:, draw_mask].sum(axis=1),
    joint_norm[:, away_win_mask].sum(axis=1),
], axis=1)

# --- 1X2 del XGBoost Classifier + ensemble ---
proba_xgb_test = xgb_clf_tuned.predict_proba(X_test)
proba_ensemble_test = WEIGHT_XGB_CLASSIFIER * proba_xgb_test + WEIGHT_POISSON * proba_poisson_test
proba_ensemble_test = proba_ensemble_test / proba_ensemble_test.sum(axis=1, keepdims=True)

y_pred_ensemble = proba_ensemble_test.argmax(axis=1)

accuracy_1x2 = accuracy_score(y_test, y_pred_ensemble)
log_loss_1x2 = log_loss(y_test, proba_ensemble_test, labels=[0, 1, 2])

print(f"Accuracy 1X2 (ensemble):  {accuracy_1x2:.4f}")
print(f"Log loss 1X2 (ensemble):  {log_loss_1x2:.4f}\n")

print("Classification report (1X2, ensemble):")
print(classification_report(y_test, y_pred_ensemble,
                             target_names=["Gana local", "Empate", "Gana visitante"], zero_division=0))

cm = confusion_matrix(y_test, y_pred_ensemble)
cm_df = pd.DataFrame(cm, index=["Real: local", "Real: empate", "Real: visitante"],
                      columns=["Pred: local", "Pred: empate", "Pred: visitante"])
print("Matriz de confusion (1X2, ensemble):")
print(cm_df)

# --- Marcador exacto: prediccion = celda con mayor probabilidad de la matriz Poisson ---
flat_argmax = np.argmax(joint_norm.reshape(len(joint_norm), -1), axis=1)
pred_home_goals = flat_argmax // (MAX_GOALS + 1)
pred_away_goals = flat_argmax % (MAX_GOALS + 1)

actual_home_goals = test_df["home_score"].values
actual_away_goals = test_df["away_score"].values

accuracy_scoreline = np.mean((pred_home_goals == actual_home_goals) & (pred_away_goals == actual_away_goals))

# Para leer la probabilidad del marcador REAL en la matriz, se acota el indice a
# MAX_GOALS (marcadores por encima de eso son rarisimos; esto es solo para poder
# indexar la matriz, no afecta la prediccion del modelo).
idx_h = np.clip(actual_home_goals, 0, MAX_GOALS)
idx_a = np.clip(actual_away_goals, 0, MAX_GOALS)
prob_correct_scoreline = joint_norm[np.arange(len(test_df)), idx_h, idx_a]
mean_proba_correct_scoreline = prob_correct_scoreline.mean()

proba_correct_1x2 = proba_ensemble_test[np.arange(len(test_df)), y_test]
mean_proba_correct_1x2 = proba_correct_1x2.mean()

print(f"\nAccuracy marcador exacto:              {accuracy_scoreline:.4f}")
print(f"Probabilidad media resultado correcto:  {mean_proba_correct_1x2:.4f}")
print(f"Probabilidad media marcador correcto:   {mean_proba_correct_scoreline:.4f}")

evaluation_metrics_summary = {
    "accuracy_1x2": float(accuracy_1x2),
    "log_loss_1x2": float(log_loss_1x2),
    "accuracy_scoreline": float(accuracy_scoreline),
    "mean_proba_correct_1x2": float(mean_proba_correct_1x2),
    "mean_proba_correct_scoreline": float(mean_proba_correct_scoreline),
}


In [ ]:
# Matriz de confusion 1X2 (grafica)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Local", "Empate", "Visitante"],
            yticklabels=["Local", "Empate", "Visitante"])
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de confusion 1X2 - Modelo 3 (ensemble)")
plt.tight_layout()
plt.show()


## 11. Predicción de partidos futuros: features, validación y funciones auxiliares

**Qué hace:** define las funciones necesarias para proyectar un partido futuro:

- `validate_team_name`: igual que en el Modelo 2, valida que el nombre del equipo
  exista en el dataset y sugiere nombres parecidos si no.
- `get_latest_team_form`: la forma más reciente de un equipo (últimos partidos
  disponibles), igual que en modelos anteriores.
- `create_match_features_v2`: arma el vector completo de ~187 variables para un
  partido hipotético, agregando ahora el **Elo vigente** de cada equipo
  (`elo_current`, el rating final tras todo el histórico) y el xG aproximado, además
  de las variables de forma ya existentes.

**Nota de diseño (para evitar un bug real que encontramos en el Modelo 2):** estas
funciones leen `long_df`, `WINDOWS`, `all_feature_cols`, `elo_current`, etc. como
variables globales **dentro del cuerpo de la función** (no como valores por defecto
fijados al definirlas). Así, funcionan igual si vienen del entrenamiento de esta sesión
o de `load_modelo3()` (Sección 14) en una sesión nueva.

In [ ]:
# ============================================================
# 11. VALIDACION DE EQUIPOS + CONSTRUCCION DE FEATURES PARA PARTIDOS FUTUROS
# ============================================================
def validate_team_name(team_name):
    """
    Verifica que team_name exista tal cual en el dataset (usa el long_df global
    vigente). Si no existe, sugiere nombres parecidos y lanza un error claro.
    """
    known_teams = sorted(set(long_df["team"].unique()))
    if team_name in known_teams:
        return team_name
    suggestions = difflib.get_close_matches(team_name, known_teams, n=5, cutoff=0.6)
    msg = f"El equipo '{team_name}' no existe en el dataset."
    if suggestions:
        msg += f" Quisiste decir: {suggestions}?"
    else:
        msg += " No se encontraron nombres parecidos; revisa la lista de equipos del dataset."
    raise ValueError(msg)


def get_latest_team_form(team_name, long_df):
    """
    Metricas de forma de un equipo usando sus ULTIMOS partidos disponibles
    (incluye el mas reciente, ya que se proyecta un partido futuro). Usa el
    WINDOWS/BASE_METRICS/STD_METRICS globales vigentes.
    """
    hist = long_df[long_df["team"] == team_name].sort_values("date")
    stats = {}
    for w in WINDOWS:
        recent = hist.tail(w)
        for col in BASE_METRICS:
            stats[f"{col}_mean_{w}"] = recent[col].mean() if len(recent) > 0 else 0.0
        for col in STD_METRICS:
            stats[f"{col}_std_{w}"] = recent[col].std() if len(recent) > 1 else 0.0
    return stats


def create_match_features_v2(home_team, away_team, neutral=1):
    """
    Construye el vector completo de variables (forma + fuerza de equipo) para un
    partido hipotetico, en el mismo orden que all_feature_cols.
    """
    home_stats = get_latest_team_form(home_team, long_df)
    away_stats = get_latest_team_form(away_team, long_df)

    row = {}
    for col, val in home_stats.items():
        row[f"home_{col}"] = val
    for col, val in away_stats.items():
        row[f"away_{col}"] = val
    for col in home_stats:
        row[f"diff_{col}"] = home_stats[col] - away_stats[col]

    elo_home = elo_current.get(home_team, ELO_INITIAL)
    elo_away = elo_current.get(away_team, ELO_INITIAL)
    row["elo_home_pre"] = elo_home
    row["elo_away_pre"] = elo_away
    row["elo_diff"] = elo_home - elo_away

    row["xg_home_approx"] = (home_stats["goals_for_mean_10"] + away_stats["goals_against_mean_10"]) / 2.0
    row["xg_away_approx"] = (away_stats["goals_for_mean_10"] + home_stats["goals_against_mean_10"]) / 2.0

    row["neutral"] = neutral
    row["home_advantage"] = 1 - neutral

    match_df = pd.DataFrame([row])
    for c in all_feature_cols:
        if c not in match_df.columns:
            match_df[c] = 0.0
    match_df = match_df[all_feature_cols].fillna(0.0)

    return match_df[all_feature_cols].values.astype(float)


## 12. Las 4 gráficas de referencia + `predict_match_advanced`

**Qué hace cada gráfica:**
1. **Heatmap de marcador:** la matriz de Poisson completa, en porcentaje (eje Y = goles
   del local, eje X = goles del visitante).
2. **Barras 1X2:** las 3 probabilidades del ensemble, con el porcentaje encima de cada
   barra.
3. **Top 10 marcadores (barras horizontales):** los 10 marcadores más probables de la
   matriz, ordenados de mayor a menor probabilidad, con formato `"Home X-Y Away"`.
4. **Evaluación del modelo:** un resumen fijo de las 4 métricas calculadas una sola vez
   sobre el test set completo (Sección 10) — no cambia entre partidos, es la "ficha
   técnica" del modelo que se muestra junto a cada predicción para dar contexto de
   confiabilidad.

`predict_match_advanced(home_team, away_team, neutral=1, max_goals=6)` junta todo:
valida equipos, arma features, obtiene el 1X2 del XGBoost Classifier y el derivado de
Poisson, los combina (ensemble), calcula la matriz de marcadores y el Top 10, muestra
las 4 gráficas, exporta 2 CSV, y devuelve un diccionario con todo.

In [ ]:
# ============================================================
# 12. LAS 4 GRAFICAS DE REFERENCIA
# ============================================================
def plot_score_heatmap(matrix, home_team, away_team, max_goals=6):
    """Grafica 1: heatmap de probabilidad por marcador exacto (%)."""
    plt.figure(figsize=(7, 6))
    sns.heatmap(matrix * 100, annot=True, fmt=".1f", cmap="YlOrRd",
                xticklabels=range(max_goals + 1), yticklabels=range(max_goals + 1),
                cbar_kws={"label": "%"})
    plt.xlabel(f"Goles {away_team} (away)")
    plt.ylabel(f"Goles {home_team} (home)")
    plt.title("Probabilidad por marcador (%)")
    plt.tight_layout()
    plt.show()


def plot_1x2_bars(proba, home_team, away_team):
    """Grafica 2: barras 1X2 con porcentaje encima de cada barra."""
    labels = [f"{home_team} gana", "Empate", f"{away_team} gana"]
    pct = proba * 100
    plt.figure(figsize=(6, 4))
    colors = ["#1f77b4", "#7f7f7f", "#d62728"]
    bars = plt.bar(labels, pct, color=colors)
    plt.ylabel("Probabilidad (%)")
    plt.title(f"{home_team} vs {away_team} - Probabilidad 1X2 (ensemble)")
    for b, v in zip(bars, pct):
        plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")
    plt.ylim(0, max(pct) + 15)
    plt.tight_layout()
    plt.show()


def plot_top10_scorelines(top10_df, home_team, away_team):
    """Grafica 3: Top 10 marcadores mas probables, barras horizontales descendente."""
    labels = [f"{home_team} {int(h)}-{int(a)} {away_team}"
              for h, a in zip(top10_df["home_goals"], top10_df["away_goals"])]
    values = top10_df["probabilidad_%"].values
    plt.figure(figsize=(7.5, 5))
    plt.barh(labels[::-1], values[::-1], color="#9467bd")
    plt.xlabel("Probabilidad (%)")
    plt.title(f"Top 10 marcadores mas probables: {home_team} vs {away_team}")
    for i, v in enumerate(values[::-1]):
        plt.text(v + 0.2, i, f"{v:.1f}%", va="center")
    plt.tight_layout()
    plt.show()


def plot_evaluation_summary(metrics=None):
    """Grafica 4: ficha tecnica del modelo (metricas del test set completo)."""
    metrics = metrics if metrics is not None else evaluation_metrics_summary
    labels = ["Accuracy 1X2", "Accuracy marcador\nexacto",
              "Prob. media resultado\ncorrecto", "Prob. media marcador\ncorrecto"]
    values = [metrics["accuracy_1x2"], metrics["accuracy_scoreline"],
              metrics["mean_proba_correct_1x2"], metrics["mean_proba_correct_scoreline"]]
    pct_values = [v * 100 for v in values]
    plt.figure(figsize=(7.5, 4.5))
    bars = plt.bar(labels, pct_values, color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])
    plt.ylabel("%")
    plt.title("Evaluacion del modelo (Modelo 3: XGBoost optimizado + Poisson)")
    for b, v in zip(bars, pct_values):
        plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# PREDICT_MATCH_ADVANCED: funcion final que junta todo
# ============================================================
CLASS_NAMES = ["Gana home_team", "Empate", "Gana away_team"]


def predict_match_advanced(home_team, away_team, neutral=1, max_goals=6,
                            export_csv=True, plot=True):
    """
    Predice un partido futuro combinando XGBoost Classifier (1X2) + Poisson
    (goles esperados -> matriz de marcadores -> 1X2 derivado). Devuelve un dict
    con la tabla 1X2, la clase predicha, los goles esperados, la matriz completa
    y el Top 10 de marcadores. Muestra las 4 graficas de referencia y exporta
    2 CSV (prediccion 1X2 y matriz de marcadores).
    """
    home_team = validate_team_name(home_team)
    away_team = validate_team_name(away_team)

    X_match = create_match_features_v2(home_team, away_team, neutral=neutral)

    proba_xgb = xgb_clf_tuned.predict_proba(X_match)[0]

    lambda_home = max(float(home_goals_model.predict(X_match)[0]), 1e-3)
    lambda_away = max(float(away_goals_model.predict(X_match)[0]), 1e-3)

    matrix = build_score_matrix(lambda_home, lambda_away, max_goals=max_goals)
    proba_poisson = derive_1x2_from_matrix(matrix)
    proba_final = combine_1x2(proba_xgb, proba_poisson)
    pred_class = int(np.argmax(proba_final))

    table_1x2 = pd.DataFrame({
        "resultado": [f"{home_team} gana", "Empate", f"{away_team} gana"],
        "prob_xgb_%": (proba_xgb * 100).round(2),
        "prob_poisson_%": (proba_poisson * 100).round(2),
        "prob_ensemble_%": (proba_final * 100).round(2),
    })

    top10 = get_top_n_scorelines(matrix, n=10)
    top10["marcador"] = top10.apply(
        lambda r: f"{home_team} {int(r.home_goals)}-{int(r.away_goals)} {away_team}", axis=1
    )

    elo_home = elo_current.get(home_team, ELO_INITIAL)
    elo_away = elo_current.get(away_team, ELO_INITIAL)

    print(f"\n{'=' * 72}\nPrediccion avanzada: {home_team} vs {away_team}\n{'=' * 72}")
    print(f"Elo vigente: {home_team}={elo_home:.0f} | {away_team}={elo_away:.0f} "
          f"(diferencia: {elo_home - elo_away:+.0f})")
    print(f"Goles esperados (Poisson): {home_team}={lambda_home:.2f} | {away_team}={lambda_away:.2f}\n")
    print("Probabilidades 1X2:")
    print(table_1x2.to_string(index=False))
    print(f"\nClase predicha: {pred_class} ({CLASS_NAMES[pred_class]})")
    print("\nTop 10 marcadores mas probables:")
    print(top10[["marcador", "probabilidad_%"]].to_string(index=False))

    if plot:
        plot_score_heatmap(matrix, home_team, away_team, max_goals=max_goals)
        plot_1x2_bars(proba_final, home_team, away_team)
        plot_top10_scorelines(top10, home_team, away_team)
        plot_evaluation_summary()

    score_matrix_df = pd.DataFrame(
        matrix * 100,
        index=[f"home_{i}" for i in range(max_goals + 1)],
        columns=[f"away_{j}" for j in range(max_goals + 1)],
    )

    if export_csv:
        safe_home = home_team.replace(" ", "_")
        safe_away = away_team.replace(" ", "_")
        pred_csv = f"prediction_{safe_home}_vs_{safe_away}_modelo3.csv"
        matrix_csv = f"score_matrix_{safe_home}_vs_{safe_away}.csv"
        table_1x2.to_csv(pred_csv, index=False)
        score_matrix_df.to_csv(matrix_csv)
        print(f"\nArchivos exportados: {pred_csv}, {matrix_csv}")

    return {
        "home_team": home_team,
        "away_team": away_team,
        "table_1x2": table_1x2,
        "predicted_class": pred_class,
        "predicted_label": CLASS_NAMES[pred_class],
        "expected_goals_home": lambda_home,
        "expected_goals_away": lambda_away,
        "score_matrix": matrix,
        "score_matrix_df": score_matrix_df,
        "top_10_scorelines": top10,
    }


## 13. Guardar los modelos y artifacts (local, no en Google Drive)

**Qué hace:** guarda los 3 modelos entrenados con `joblib` y un diccionario de
artifacts con todo lo necesario para reutilizar el pipeline completo sin reentrenar:
las ~187 variables usadas, el historial de equipos, los ratings Elo vigentes, los
parámetros del feature engineering, los pesos del ensemble, y las métricas de
evaluación (para poder mostrar la Gráfica 4 sin recalcular todo el test set).

In [ ]:
# ============================================================
# 13. GUARDAR MODELOS Y ARTIFACTS
# ============================================================
joblib.dump(xgb_clf_tuned, "modelo_xgboost_1x2_tuned.joblib")
joblib.dump(home_goals_model, "modelo_xgboost_home_goals.joblib")
joblib.dump(away_goals_model, "modelo_xgboost_away_goals.joblib")
print("Modelos guardados: modelo_xgboost_1x2_tuned.joblib, modelo_xgboost_home_goals.joblib, "
      "modelo_xgboost_away_goals.joblib")

artifacts_modelo3 = {
    "feature_cols": all_feature_cols,             # las ~187 variables usadas (todas, sin recorte)
    "team_history": long_df,                      # historial largo por equipo
    "elo_current": elo_current,                   # rating Elo vigente por equipo
    "elo_initial": ELO_INITIAL,
    "elo_k": ELO_K,
    "windows": WINDOWS,
    "base_metrics": BASE_METRICS,
    "std_metrics": STD_METRICS,
    "class_names": CLASS_NAMES,
    "weight_xgb_classifier": WEIGHT_XGB_CLASSIFIER,
    "weight_poisson": WEIGHT_POISSON,
    "evaluation_metrics_summary": evaluation_metrics_summary,
    "max_goals_default": MAX_GOALS,
}

with open("artifacts_modelo3.pkl", "wb") as f:
    pickle.dump(artifacts_modelo3, f)

print("Artifacts guardados en: artifacts_modelo3.pkl")


## 14. Cargar el modelo guardado (sin reentrenar)

**Qué hace:** `load_modelo3()` carga los 3 `.joblib` y el `.pkl`, y restaura las
variables globales que necesitan `predict_match_advanced` y sus funciones auxiliares
(`all_feature_cols`, `long_df`, `elo_current`, `WINDOWS`, los pesos del ensemble, las
métricas de evaluación, etc.). Como las funciones de la Sección 11-12 leen estas
variables como globales en el momento en que se **llaman** (no cuando se definen),
esta celda puede ejecutarse en una sesión nueva (después de solo correr los imports y
las definiciones de funciones) sin repetir el entrenamiento.

In [ ]:
# ============================================================
# 14. CARGAR MODELOS GUARDADOS (para usar en una sesion nueva sin reentrenar)
# ============================================================
def load_modelo3(clf_path="modelo_xgboost_1x2_tuned.joblib",
                  home_path="modelo_xgboost_home_goals.joblib",
                  away_path="modelo_xgboost_away_goals.joblib",
                  artifacts_path="artifacts_modelo3.pkl",
                  set_globals=True):
    """
    Carga los 3 modelos XGBoost y los artifacts del Modelo 3 desde disco. Si
    set_globals=True (por defecto), tambien actualiza las variables globales que
    usan predict_match_advanced y sus funciones auxiliares, para poder predecir
    sin haber corrido el entrenamiento en esta sesion.
    """
    clf = joblib.load(clf_path)
    home_model = joblib.load(home_path)
    away_model = joblib.load(away_path)
    with open(artifacts_path, "rb") as f:
        artifacts = pickle.load(f)

    if set_globals:
        global xgb_clf_tuned, home_goals_model, away_goals_model
        global all_feature_cols, long_df, WINDOWS, BASE_METRICS, STD_METRICS
        global elo_current, ELO_INITIAL, ELO_K, CLASS_NAMES
        global WEIGHT_XGB_CLASSIFIER, WEIGHT_POISSON, evaluation_metrics_summary, MAX_GOALS

        xgb_clf_tuned = clf
        home_goals_model = home_model
        away_goals_model = away_model
        all_feature_cols = artifacts["feature_cols"]
        long_df = artifacts["team_history"]
        elo_current = artifacts["elo_current"]
        ELO_INITIAL = artifacts["elo_initial"]
        ELO_K = artifacts["elo_k"]
        WINDOWS = artifacts["windows"]
        BASE_METRICS = artifacts["base_metrics"]
        STD_METRICS = artifacts["std_metrics"]
        CLASS_NAMES = artifacts["class_names"]
        WEIGHT_XGB_CLASSIFIER = artifacts["weight_xgb_classifier"]
        WEIGHT_POISSON = artifacts["weight_poisson"]
        evaluation_metrics_summary = artifacts["evaluation_metrics_summary"]
        MAX_GOALS = artifacts["max_goals_default"]

    print(f"Modelo 3 cargado: {clf_path}, {home_path}, {away_path}, {artifacts_path}")
    print(f"Variables restauradas: {len(artifacts['feature_cols'])}")
    return clf, home_model, away_model, artifacts


# Ejemplo de uso en una sesion nueva (descomenta para usarlo en lugar de reentrenar):
# xgb_clf_tuned, home_goals_model, away_goals_model, artifacts_modelo3 = load_modelo3()


## 15. Ejemplos de predicción

Los 4 partidos solicitados. Cada llamada imprime las tablas, genera las 4 gráficas de
referencia, y exporta los 2 CSV correspondientes.

In [ ]:
# Ejemplo 1: partido objetivo del proyecto
result_france_paraguay = predict_match_advanced("France", "Paraguay", neutral=1)


In [ ]:
# Ejemplo 2
result_mexico_england = predict_match_advanced("Mexico", "England", neutral=1)


In [ ]:
# Ejemplo 3
result_brazil_norway = predict_match_advanced("Brazil", "Norway", neutral=1)


In [ ]:
# Ejemplo 4
result_portugal_spain = predict_match_advanced("Portugal", "Spain", neutral=1)


## Resumen: qué cambió respecto al Modelo 2 (XGBoost Vanilla)

| Aspecto | Modelo 2 — Vanilla | Modelo 3 — Optimizado + Poisson |
|---|---|---|
| Feature engineering base | ~180 variables de forma (rolling 3/5/10/15/20) | **Idéntico**, sin cambios |
| Variables de fuerza de equipo | Ninguna | **Elo histórico** (sin leakage), `elo_diff`, xG aproximado, ventaja local/neutral (+7 variables → ~187 en total) |
| Modelo 1X2 | XGBoost vanilla (defaults) | XGBoost **optimizado** (`RandomizedSearchCV` + `TimeSeriesSplit`) |
| Marcador exacto | No disponible | Matriz de Poisson (goles esperados por 2 `XGBRegressor` con `count:poisson`) |
| Salida 1X2 | Solo XGBoost Classifier | **Ensemble** ponderado XGBoost + Poisson |
| Evaluación | Accuracy, log loss, F1, matriz de confusión | Todo lo anterior **+ accuracy de marcador exacto + probabilidad media del resultado/marcador correcto** |
| Francia vs Paraguay | Favorecía a Paraguay (sin señal de fuerza histórica) | Favorece a Francia, consistente con la brecha de Elo (~275 puntos) |

**Limitación honesta:** este modelo es más robusto porque ahora distingue la fuerza
real de cada selección, pero el fútbol conserva varianza irreducible — la probabilidad
del marcador exacto más probable normalmente ronda el 10-13%, no un valor cercano al
100%. Eso es esperado y correcto: significa que el modelo no está sobreconfiado en un
resultado que, en la práctica, tiene muchos desenlaces plausibles.
